
# Thí nghiệm 02 — Huấn luyện dự báo áp suất bề mặt CMIP6

## 1. Thiết lập dữ liệu và môi trường

### 1.1. Mục tiêu

Notebook này triển khai huấn luyện các mô hình dự báo chuỗi thời gian trên dữ liệu áp suất bề mặt (Surface Pressure) thuộc bộ dữ liệu khí hậu CMIP6/CORDEX.

Bốn mô hình được sử dụng:

1. LSTM.
2. PatchTST kết hợp toán tử Koopman.
3. Autoformer kết hợp toán tử Koopman.
4. Informer kết hợp toán tử Koopman.

Thực nghiệm gồm hai nhóm:

- **Dự báo tại 5 vị trí:** huấn luyện bốn mô hình với cấu hình cố định nhằm tạo dữ liệu cho Figure 6.
- **Grid Search:** khảo sát ảnh hưởng của chân trời dự báo và kích thước biểu diễn tiềm ẩn nhằm tạo dữ liệu cho Figure 5.

Hai nhóm huấn luyện được lưu vào những thư mục checkpoint riêng và không được gộp kết quả.

### 1.2. Nguồn dữ liệu

Dữ liệu đầu vào được chuẩn bị trong:

`04_climate_pressure_data.ipynb`

Tệp sử dụng:

`data/pressure_nearest_5sites.pt`

Tensor `pressure_raw` có kích thước:

$$
P_{\mathrm{raw}}\in\mathbb{R}^{400\times5}
$$

Trong đó:

- 400 là số thời điểm.
- 5 là số vị trí không gian.
- Đơn vị giá trị gốc là Pascal (Pa).

Các vị trí tương ứng với chỉ số:

`[0, 5000, 10000, 15000, 20000]`

### 1.3. Chuẩn hóa dữ liệu

Notebook sử dụng `MinMaxScaler` để chuẩn hóa từng kênh trong toàn bộ 400 thời điểm:

$$
x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}
$$

Bộ chuẩn hóa được khớp trên chính đoạn dữ liệu được sử dụng để huấn luyện.

Do đó, kết quả đánh giá của nhánh này là đánh giá trên dữ liệu huấn luyện, không phải trên tập kiểm tra độc lập.

### 1.4. Tạo cửa sổ trượt

Với độ dài đầu vào $P$ và chân trời dự báo $H$:

$$
X_i=P_{\mathrm{scaled}}[i:i+P]
$$

$$
Y_i=P_{\mathrm{scaled}}[i+P:i+P+H]
$$

Nhãn dự báo được chuyển vị và làm phẳng thành vector gồm $5H$ phần tử.

Số cửa sổ được tạo:

$$
N=400-P-H
$$

Ví dụ, khi $P=120$ và $H=15$:

$$
N=265
$$

Kích thước dữ liệu:

- Input: `(265, 120, 5)`.
- Target: `(265, 75)`.

### 1.5. Quản lý kết quả

Các checkpoint của hai nhóm thực nghiệm được lưu riêng:

- `results/pressure_5channels_figure6_e4000/`
- `results/pressure_5channels_grid_e4000/`

Notebook giữ mã nguồn huấn luyện và cơ chế lưu kết quả để phục vụ khả năng tái lập.

Trong quá trình tái cấu trúc, không huấn luyện lại mô hình hoặc ghi đè checkpoint đã có.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler


# 1. Xác định thư mục gốc của project
project_root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (
            path / "experiments" / "02_climate"
            / "data" / "pressure_nearest_5sites.pt"
        ).is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy pressure_nearest_5sites.pt"
    )


# 2. Xác định đường dẫn các tệp
experiment_dir = project_root / "experiments" / "02_climate"

data_path = experiment_dir / "data" / "pressure_nearest_5sites.pt"
model_notebook = experiment_dir / "02_climate_model.ipynb"

# Thư mục dành riêng cho 4 checkpoint của Figure 6
results_dir = (
    experiment_dir
    / "results"
    / "pressure_5channels_figure6_e4000"
)

results_dir.mkdir(parents=True, exist_ok=True)


# 3. Nạp dữ liệu áp suất đã trích xuất
dataset = torch.load(
    data_path,
    map_location="cpu",
    weights_only=True
)

pressure_raw = dataset["pressure_raw"].numpy()
all_indices = dataset["location_indices"]

location_indices = [0, 5000, 10000, 15000, 20000]

assert all_indices == location_indices

raw = pressure_raw.copy()
input_dim = raw.shape[1]

assert raw.shape == (400, 5)
assert input_dim == 5


# 4. Cấu hình dự báo tại 5 vị trí — Figure 6
patch_lengths = [120]
horizons = [15]
d_models = [8]

epochs = 4000
learning_rate = 3e-4
lyap_weight = 0.1
seed = 7

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 5. Hàm tạo các cửa sổ trượt
def build_pressure_dataset(raw, patch_len, horizon):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(raw[:400])

    X, Y = [], []

    for i in range(len(scaled) - patch_len - horizon):
        X.append(scaled[i:i + patch_len])
        Y.append(
            scaled[i + patch_len:i + patch_len + horizon]
            .T.reshape(-1)
        )

    x_in = torch.tensor(
        np.stack(X), dtype=torch.float32
    )
    x_out = torch.tensor(
        np.stack(Y), dtype=torch.float32
    )

    return x_in, x_out, scaler


# 6. Kiểm tra kích thước dữ liệu tại P=120, H=15
x_in, x_out, scaler = build_pressure_dataset(
    raw, patch_len=120, horizon=15
)


# 7. Tổng hợp cấu hình
print("=== CMIP6 SURFACE PRESSURE ===")
print("Device:", device)
print("Dataset:", data_path)
print("Model notebook:", model_notebook)

print("Raw:", raw.shape)
print("X:", x_in.shape)
print("Y:", x_out.shape)

print("Epochs:", epochs)
print(
    "Total runs:",
    len(patch_lengths)
    * len(horizons)
    * len(d_models)
    * 4
)

print("Results:", results_dir)

c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


=== KAGGLE SURFACE PRESSURE ===
GPU: Quadro T1000
Dataset: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\pressure_nearest_5sites.pt
Model notebook: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\02_climate_model.ipynb
Raw: (400, 5)
X: torch.Size([265, 120, 5])
Y: torch.Size([265, 75])
Epochs: 4000
Total runs: 4
Results: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\results\pressure_5channels_figure6_e4000



## 2. Nạp kiến trúc mô hình

### 2.1. Mục tiêu

Các lớp kiến trúc được sử dụng chung từ:

`02_climate_model.ipynb`

Notebook hiện tại không định nghĩa lại các backbone hoặc toán tử Koopman.

Thay vào đó, mã nguồn trích xuất các thành phần cần thiết từ notebook Model.

### 2.2. Phương pháp nạp định nghĩa

Notebook Model được đọc dưới dạng JSON.

Với mỗi code cell, sử dụng `ast.parse()` để phân tích cú pháp và trích xuất:

- Câu lệnh import.
- Định nghĩa hàm.
- Định nghĩa lớp.

Các thành phần này được biên dịch và nạp vào môi trường thực thi hiện tại.

Cách làm này cho phép sử dụng các kiến trúc mà không chạy những cell khác trong notebook Model.

### 2.3. Các mô hình

| Mô hình | Vai trò |
|---|---|
| `SimpleLSTMForecaster` | Mô hình đối chứng |
| `Koopformer_PatchTST` | PatchTST kết hợp Koopman |
| `Koopformer_Autoformer` | Autoformer kết hợp Koopman |
| `Koopformer_Informer` | Informer kết hợp Koopman |

Hàm `create_model()` nhận các tham số:

- Tên mô hình.
- Độ dài đầu vào.
- Chân trời dự báo.
- Kích thước biểu diễn `d_model`.

Hàm trả về mô hình tương ứng với cấu hình được chỉ định.

### 2.4. Bảo toàn kiến trúc

Tất cả các lớp mô hình được giữ theo mã nguồn đã sử dụng trong quá trình huấn luyện.

Không thay đổi tên tham số, kích thước tầng, cách khởi tạo hoặc phương thức `forward()`.

Điều này cần thiết để duy trì khả năng nạp các checkpoint đã lưu.


In [ ]:
import ast
import json
from pathlib import Path


with open(model_notebook, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Extract definitions without executing dataset/training code
allowed_nodes = (
    ast.Import,
    ast.ImportFrom,
    ast.ClassDef,
    ast.FunctionDef,
    ast.AsyncFunctionDef
)

definitions = []
class_names = []

for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = cell["source"]
    if isinstance(source, list):
        source = "".join(source)

    if not source.strip():
        continue

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, allowed_nodes):
            definitions.append(node)

            if isinstance(node, ast.ClassDef):
                class_names.append(node.name)

# Execute only imports, classes and functions
module = ast.Module(body=definitions, type_ignores=[])
ast.fix_missing_locations(module)

exec(
    compile(module, str(model_notebook), "exec"),
    globals()
)

# Model factory
def create_model(model_name, patch_len, horizon, d_model):
    if model_name == "LSTM":
        return SimpleLSTMForecaster(
            input_dim, d_model, 2, horizon
        )

    if model_name == "PatchTST":
        return Koopformer_PatchTST(
            input_dim, patch_len, horizon,
            patch_len=patch_len, d_model=d_model
        )

    if model_name == "Autoformer":
        return Koopformer_Autoformer(
            input_dim, patch_len, horizon,
            patch_len=patch_len, d_model=d_model
        )

    if model_name == "Informer":
        return Koopformer_Informer(
            input_dim, patch_len, horizon,
            patch_len=patch_len, d_model=d_model
        )

    raise ValueError(f"Unknown model: {model_name}")


=== LOADED MODEL DEFINITIONS ===
Classes found: ['StrictStableKoopmanOperator', 'SinCosPosEnc', 'PatchEmbed1D', 'SeriesDecomp', 'PatchTST_Backbone', 'Koopformer_PatchTST', 'SimpleAutoformer', 'Koopformer_Autoformer', 'InformerSparse', 'Koopformer_Informer', 'SimpleLSTMForecaster']
LSTM         Parameters: 1,731
PatchTST     Parameters: 11,507
Autoformer   Parameters: 15,111
Informer     Parameters: 11,499

All four models initialized successfully.



## 3. Hàm huấn luyện và chính quy hóa Lyapunov

### 3.1. Mục tiêu

Hàm `train_pressure_model()` thực hiện huấn luyện một mô hình trên các cửa sổ dữ liệu Surface Pressure đã được chuẩn hóa.

Hàm sử dụng cùng một quy trình tối ưu cho bốn mô hình để bảo đảm tính nhất quán trong Grid Search.

### 3.2. Hàm mất mát dự báo

Sai số dự báo được tính bằng Mean Squared Error:

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat y_i-y_i)^2
$$

Trong đó $\hat y_i$ là giá trị dự báo và $y_i$ là nhãn thực tế.

### 3.3. Chính quy hóa Lyapunov

Đối với các mô hình sử dụng Koopman, hàm mất mát bổ sung thành phần chính quy hóa nhằm hạn chế sự gia tăng năng lượng của biểu diễn tiềm ẩn.

Gọi $z$ và $z'$ lần lượt là biểu diễn trước và sau toán tử Koopman.

Ma trận Lyapunov được chọn:

$$
P=I
$$

Hàm năng lượng:

$$
V(z)=z^\top Pz
$$

Thành phần phạt:

$$
\mathcal{L}_{\mathrm{Lyap}}
=
\frac{1}{B}
\sum_{i=1}^{B}
\max(0,V(z'_i)-V(z_i))
$$

Trong đó $B$ là số mẫu được sử dụng ở mỗi lượt cập nhật.

### 3.4. Hàm mục tiêu tổng hợp

$$
\mathcal{L}
=
\mathcal{L}_{\mathrm{MSE}}
+
\lambda\mathcal{L}_{\mathrm{Lyap}}
$$

Hệ số:

- LSTM: $\lambda=0$.
- Các biến thể Koopman: $\lambda=0.1$.

LSTM không có phép biến đổi Koopman nên chỉ tối ưu sai số dự báo.

### 3.5. Quy trình tối ưu

Hàm sử dụng:

- Bộ tối ưu Adam.
- Learning rate bằng `3e-4`.
- 4000 epoch cho mỗi cấu hình.
- Full-batch training.
- Hạt giống ngẫu nhiên được thiết lập từ cấu hình thực nghiệm.

Mỗi epoch thực hiện truyền xuôi, tính loss, lan truyền ngược và cập nhật trọng số.

Sau khi hoàn thành huấn luyện, mô hình được chuyển sang chế độ `eval()` để tạo dự báo trên chính dữ liệu đã dùng huấn luyện.

### 3.6. Lịch sử được ghi nhận

Hàm trả về ba thành phần:

| Thành phần | Ý nghĩa |
|---|---|
| `predictions` | Kết quả dự báo sau huấn luyện |
| `losses` | Giá trị hàm mất mát theo epoch |
| `eigs` | Chỉ số theo dõi tham số toán tử Koopman |

Trong mã nguồn, `eigs` được tính từ giá trị lớn nhất của `sigmoid(S_raw)`.

Đây không phải phép tính trực tiếp trị riêng lớn nhất của ma trận Koopman và cũng chưa nhân hệ số giới hạn 0.99.

Vì vậy, khi trực quan hóa hoặc phân tích, cần sử dụng tên đại lượng chính xác.

### 3.7. Phạm vi đánh giá

Các dự báo cuối cùng được tạo trên tập cửa sổ đã dùng trong quá trình tối ưu.

Những metric tính từ kết quả này phản ánh mức độ khớp dữ liệu huấn luyện, chưa đánh giá khả năng tổng quát hóa trên tập Test độc lập.

In [ ]:
import torch.nn as nn
import torch.optim as optim

def set_seed(seed=7):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(seed)

def _nested_getattr(obj, path):
    for p in path.split("."):
        obj = getattr(obj, p, None)
        if obj is None:
            return None
    return obj

def train_pressure_model(
    model, x_in, x_out,
    epochs=4000, lr=3e-4,
    koop_attr="koop", lyap_weight=0.1
):
    model = model.to(device)
    x_in = x_in.to(device)
    x_out = x_out.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()

    losses, eigs = [], []

    # Khởi tạo ma trận Lyapunov P = I
    with torch.no_grad():
        try:
            _, z, z_next = model(
                x_in[:1], return_latents=True
            )
        except ValueError:
            z = z_next = torch.zeros(
                1, model.fc.in_features, device=device
            )

    P = torch.eye(z.shape[1], device=device)

    for ep in range(epochs):
        model.train()
        optimizer.zero_grad()

        pred, z, z_next = model(
            x_in, return_latents=True
        )

        pred_loss = mse(pred, x_out)

        if lyap_weight:
            zp = torch.einsum("bi,ij->bj", z, P)
            znp = torch.einsum("bi,ij->bj", z_next, P)

            lyap = torch.relu(
                (znp * z_next).sum(1)
                - (zp * z).sum(1)
            ).mean()
        else:
            lyap = torch.zeros(1, device=device)

        loss = pred_loss + lyap_weight * lyap

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        obj = (
            _nested_getattr(model, koop_attr)
            if koop_attr else None
        )

        eig_raw = (
            obj.S_raw
            if isinstance(obj, StrictStableKoopmanOperator)
            else None
        )

        eigs.append(
            0.0 if eig_raw is None
            else torch.sigmoid(eig_raw).abs().max().item()
        )

        if ep % max(1, epochs // 10) == 0 or ep == epochs - 1:
            print(
                f"Epoch {ep:>4}/{epochs} "
                f"MSE {pred_loss.item():.5f} "
                f"Lyap {(lyap_weight * lyap).item():.5f}"
            )

    # Tạo dự báo cuối cùng trên các cửa sổ huấn luyện
    model.eval()

    with torch.no_grad():
        predictions = model(x_in).cpu().numpy()

    return predictions, np.array(losses), np.array(eigs)



## 4. Huấn luyện bốn mô hình cho dự báo Surface Pressure

### 4.1. Mục tiêu

Nhóm thực nghiệm đầu tiên huấn luyện bốn mô hình với một cấu hình cố định trên dữ liệu áp suất tại năm vị trí.

Các kết quả được sử dụng để dựng biểu đồ dự báo Surface Pressure, tương ứng với Figure 6 trong hệ thống hình minh họa của thực nghiệm.

### 4.2. Cấu hình

| Tham số | Giá trị |
|---|---|
| Số kênh | 5 |
| Độ dài đầu vào | 120 |
| Chân trời dự báo | 15 |
| `d_model` | 8 |
| Epoch | 4000 |
| Learning rate | 0.0003 |
| Random seed | 7 |

Tổng số lượt huấn luyện:

$$
1\times1\times1\times4=4
$$

Với 400 thời điểm, số cửa sổ mỗi cấu hình:

$$
400-120-15=265
$$

Input có kích thước `(265, 120, 5)` và target có kích thước `(265, 75)`.

### 4.3. Huấn luyện và checkpoint

Mỗi mô hình được khởi tạo theo cấu hình cố định, sau đó huấn luyện 4000 epoch.

Sau huấn luyện, notebook tính:

- MSE trên các cửa sổ huấn luyện.
- MAE trên các cửa sổ huấn luyện.

Checkpoint lưu:

- Trọng số cuối cùng.
- Dự báo và nhãn thực tế.
- Lịch sử loss.
- Lịch sử chỉ số theo dõi Koopman.
- Các metric và siêu tham số.
- Trạng thái ngẫu nhiên phục vụ việc tiếp tục thực nghiệm.

### 4.4. Thư mục kết quả

Các kết quả được lưu tại:

`results/pressure_5channels_figure6_e4000/`

Tên checkpoint có dạng:

`{model}_p120_h15_d8.pt`

Các metric được tổng hợp vào `metrics.csv`.

### 4.5. Kết quả đã lưu

Theo tệp `metrics.csv` của thư mục Figure 6 trên GitHub:

| Mô hình | MSE | MAE |
|---|---:|---:|
| LSTM | 0.012755 | 0.089925 |
| PatchTST | 0.008960 | 0.074818 |
| Autoformer | 0.006107 | 0.058700 |
| Informer | 0.007951 | 0.070404 |

Các số liệu này thuộc đúng bốn checkpoint của nhánh `pressure_5channels_figure6_e4000`.

Không sử dụng metric của `pressure_grid_search_e4000` để thay thế, vì hai thư mục có kết quả khác nhau dù cùng cấu hình.

### 4.6. Giới hạn diễn giải

Các metric trên được tính trên dữ liệu huấn luyện.

Nhánh thực nghiệm này không sử dụng tập kiểm tra độc lập.

Vì vậy, kết quả chỉ phản ánh chất lượng khớp dữ liệu của bốn mô hình theo cấu hình đã lựa chọn.


In [ ]:
import gc
import itertools

model_names = ["LSTM", "PatchTST", "Autoformer", "Informer"]
figure6_metrics_path = results_dir / "metrics.csv"

# Figure 6: 4 configurations (5-channel Surface Pressure)
configs = list(itertools.product(
    patch_lengths,
    horizons,
    d_models,
    model_names
))

if device.type != "cuda":
    raise RuntimeError(
        "Grid Search requires a CUDA GPU. "
        "Please enable CUDA on the local GPU."
    )

results_dir.mkdir(parents=True, exist_ok=True)

# Fixed seed for reproducible sequential training
set_seed(seed)

figure6_records = {}

def checkpoint_path(p, h, d, name):
    return results_dir / f"{name.lower()}_p{p}_h{h}_d{d}.pt"

def save_metrics():
    df = pd.DataFrame(
        list(figure6_records.values()),
        columns=[
            "Model", "PatchLen", "Horizon",
            "d_model", "MSE", "MAE"
        ]
    )
    temp_path = figure6_metrics_path.with_suffix(".csv.tmp")
    df.to_csv(temp_path, index=False)
    temp_path.replace(figure6_metrics_path)

# Restore completed configurations
completed = 0

for p, h, d, name in configs:
    path = checkpoint_path(p, h, d, name)

    if not path.exists():
        break

    ckpt = torch.load(
        path, map_location="cpu", weights_only=True
    )

    assert ckpt["epochs"] == epochs
    assert ckpt["patch_len"] == p
    assert ckpt["horizon"] == h
    assert ckpt["d_model"] == d
    assert ckpt["model"] == name

    figure6_records[(p, h, d, name)] = {
        "Model": name,
        "PatchLen": p,
        "Horizon": h,
        "d_model": d,
        "MSE": ckpt["mse"],
        "MAE": ckpt["mae"]
    }

    completed += 1

    # Restore random state from last completed run
    torch.set_rng_state(ckpt["rng_torch"])

    if ckpt["rng_cuda"]:
        torch.cuda.set_rng_state_all(ckpt["rng_cuda"])

# Avoid silently continuing from inconsistent checkpoints
for p, h, d, name in configs[completed:]:
    if checkpoint_path(p, h, d, name).exists():
        raise RuntimeError(
            "Non-sequential checkpoints detected. "
            "Check the results directory before resuming."
        )

save_metrics()

print("=== SURFACE PRESSURE GRID SEARCH ===")
print("Device:", device)
print("Epochs per model:", epochs)
print("Total configurations:", len(configs))
print("Completed:", completed)
print("Remaining:", len(configs) - completed)
print("Results:", results_dir)

# Train remaining configurations
for idx, (p, h, d, name) in enumerate(
    configs[completed:], start=completed + 1
):
    print(
        f"\n[{idx}/{len(configs)}] "
        f"{name} | P={p}, H={h}, d_model={d}"
    )

    x_train, y_train, _ = build_pressure_dataset(
        raw, patch_len=p, horizon=h
    )

    model = create_model(
        name, patch_len=p, horizon=h, d_model=d
    )

    is_lstm = name == "LSTM"

    pred, loss, eig = train_pressure_model(
        model,
        x_train,
        y_train,
        epochs=epochs,
        lr=learning_rate,
        koop_attr=None if is_lstm else "koop",
        lyap_weight=0.0 if is_lstm else lyap_weight
    )

    truth = y_train.numpy()
    error = pred - truth

    mse_value = float(np.mean(error ** 2))
    mae_value = float(np.mean(np.abs(error)))

    # Save final model, predictions and training curves
    ckpt = {
        "model": name,
        "model_state_dict": model.cpu().state_dict(),
        "pred": torch.from_numpy(pred.copy()),
        "truth": torch.from_numpy(truth.copy()),
        "loss": torch.from_numpy(loss.copy()),
        "eig": torch.from_numpy(eig.copy()),
        "mse": mse_value,
        "mae": mae_value,
        "patch_len": p,
        "horizon": h,
        "d_model": d,
        "input_dim": input_dim,
        "epochs": epochs,
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all()
    }

    path = checkpoint_path(p, h, d, name)
    temp_path = path.with_suffix(".pt.tmp")

    torch.save(ckpt, temp_path)
    temp_path.replace(path)

    # Update metrics after every completed run
    figure6_records[(p, h, d, name)] = {
        "Model": name,
        "PatchLen": p,
        "Horizon": h,
        "d_model": d,
        "MSE": mse_value,
        "MAE": mae_value
    }

    save_metrics()

    print(
        f"Saved {name}: "
        f"MSE={mse_value:.6f}, "
        f"MAE={mae_value:.6f}"
    )
    
    del model, x_train, y_train, pred, loss, eig, ckpt
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== GRID SEARCH COMPLETE ===")
print("Completed runs:", len(figure6_records))
print("Metrics saved:", figure6_metrics_path)


=== SURFACE PRESSURE GRID SEARCH ===
Device: cuda
Epochs per model: 4000
Total configurations: 4
Completed: 0
Remaining: 4
Results: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\results\pressure_5channels_figure6_e4000

[1/4] LSTM | P=120, H=15, d_model=8
Epoch    0/4000 MSE 0.36031 Lyap 0.00000
Epoch  400/4000 MSE 0.05570 Lyap 0.00000
Epoch  800/4000 MSE 0.03623 Lyap 0.00000
Epoch 1200/4000 MSE 0.02791 Lyap 0.00000
Epoch 1600/4000 MSE 0.02555 Lyap 0.00000
Epoch 2000/4000 MSE 0.02385 Lyap 0.00000
Epoch 2400/4000 MSE 0.02263 Lyap 0.00000
Epoch 2800/4000 MSE 0.01671 Lyap 0.00000
Epoch 3200/4000 MSE 0.01509 Lyap 0.00000
Epoch 3600/4000 MSE 0.01385 Lyap 0.00000
Epoch 3999/4000 MSE 0.01276 Lyap 0.00000
Saved LSTM: MSE=0.012755, MAE=0.089925
Checkpoint saved to: C:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\results\pressure_5channels_figure6_e4000\lstm_p120_h15_d8.pt

[2/4] PatchTST | P=120, H=15, d_model=8
Epoch    0/4000 MSE 0.40512 Lyap 0.00000
Epoch  400/4000 MSE 0.


## 5. Grid Search trên dữ liệu Surface Pressure

### 5.1. Mục tiêu

Nhóm thực nghiệm thứ hai khảo sát ảnh hưởng của chân trời dự báo và kích thước biểu diễn tiềm ẩn đến sai số dự báo Surface Pressure.

Các kết quả được sử dụng để tổng hợp bảng đánh giá và xây dựng heatmap MSE/MAE.

Khác với nhánh Figure 6, Grid Search sử dụng nhiều cấu hình và lưu checkpoint vào một thư mục riêng.

### 5.2. Không gian tìm kiếm

Độ dài đầu vào được cố định:

$$
P=120
$$

Các chân trời dự báo:

$$
H\in\{10,15,20,25,30\}
$$

Các giá trị kích thước biểu diễn:

$$
d_{\mathrm{model}}\in\{8,16,24,32,40,48\}
$$

Với bốn mô hình, tổng số lượt huấn luyện:

$$
1\times5\times6\times4=120
$$

Mỗi cấu hình sử dụng 4000 epoch.

### 5.3. Các mô hình được khảo sát

Grid Search sử dụng:

- LSTM.
- PatchTST kết hợp Koopman.
- Autoformer kết hợp Koopman.
- Informer kết hợp Koopman.

Các kiến trúc được khởi tạo bằng hàm `create_model()` đã định nghĩa ở mục 2.

Giá trị `d_model` được truyền vào từng mô hình theo cách triển khai hiện tại.

Đối với LSTM, giá trị này được dùng làm `hidden_dim`.

Đối với các biến thể Transformer, giá trị được sử dụng trong các lớp backbone tương ứng.

Lưu ý rằng kích thước toán tử Koopman của biến thể Autoformer phụ thuộc vào `horizon`, theo kiến trúc đã định nghĩa trong notebook Model.

### 5.4. Quy trình huấn luyện

Với mỗi cấu hình:

1. Tạo cửa sổ trượt từ dữ liệu áp suất đã chuẩn hóa.
2. Khởi tạo mô hình tương ứng.
3. Huấn luyện 4000 epoch.
4. Tạo dự báo trên các cửa sổ huấn luyện.
5. Tính MSE và MAE.
6. Lưu checkpoint, lịch sử huấn luyện và các metric.

Các mô hình có checkpoint đã lưu được bỏ qua nhằm hỗ trợ tiếp tục thực nghiệm.

### 5.5. Quản lý kết quả

Thư mục:

`results/pressure_5channels_grid_e4000/`

Tên checkpoint:

`{model}_p{patch_length}_h{horizon}_d{d_model}.pt`

Ví dụ:

`patchtst_p120_h15_d48.pt`

Tệp `metrics.csv` chứa sáu cột:

| Cột | Nội dung |
|---|---|
| `Model` | Tên mô hình |
| `PatchLen` | Độ dài đầu vào |
| `Horizon` | Chân trời dự báo |
| `d_model` | Kích thước biểu diễn |
| `MSE` | Sai số bình phương trung bình |
| `MAE` | Sai số tuyệt đối trung bình |

### 5.6. Tình trạng kết quả

Tệp `metrics.csv` đã lưu chứa 120 dòng kết quả.

Tuy nhiên, số checkpoint hiện diện trên GitHub chưa khớp với số dòng trong CSV.

Vì vậy, không được kết luận rằng toàn bộ 120 checkpoint đã được đồng bộ lên repository.

Việc xác minh số lượng checkpoint còn thiếu sẽ được thực hiện trong giai đoạn kiểm toán Results.

### 5.7. Giới hạn đánh giá

Các chỉ số MSE và MAE được tính trên dữ liệu đã sử dụng để huấn luyện.

Chúng không tương đương với sai số trên một tập Test độc lập.

Mục đích của Grid Search là phân tích sự thay đổi của kết quả theo cấu hình thực nghiệm đã triển khai.


In [ ]:
import gc
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# 1. Separate output directory for full grid search
grid_results_dir = (
    experiment_dir
    / "results"
    / "pressure_5channels_grid_e4000"
)
grid_results_dir.mkdir(parents=True, exist_ok=True)

grid_metrics_path = grid_results_dir / "metrics.csv"

# 2. Full heatmap search space
grid_patch_lengths = [120]
grid_horizons = [10, 15, 20, 25, 30]
grid_d_models = [8, 16, 24, 32, 40, 48]
grid_model_names = ["LSTM", "PatchTST", "Autoformer", "Informer"]

grid_configs = list(itertools.product(
    grid_patch_lengths,
    grid_horizons,
    grid_d_models,
    grid_model_names
))

if device.type != "cuda":
    raise RuntimeError(
        "Heatmap grid search requires CUDA. "
        "Please enable CUDA on the local GPU."
    )

assert raw.shape == (400, 5)
assert input_dim == 5

# 3. Fixed seed for reproducible sequential training
set_seed(seed)

# 4. Helper paths / save functions
def grid_checkpoint_path(p, h, d, name):
    return grid_results_dir / f"{name.lower()}_p{p}_h{h}_d{d}.pt"

def save_grid_metrics(records_dict):
    if len(records_dict) == 0:
        df = pd.DataFrame(columns=[
            "Model", "PatchLen", "Horizon",
            "d_model", "MSE", "MAE"
        ])
    else:
        df = pd.DataFrame(list(records_dict.values()))

    temp_path = grid_metrics_path.with_suffix(".csv.tmp")
    df.to_csv(temp_path, index=False)
    temp_path.replace(grid_metrics_path)

# 5. Load already completed runs (if any)
grid_records[(p, h, d, name)] = {
    "Model": name,
    "PatchLen": p,
    "Horizon": h,
    "d_model": d,
    "MSE": ckpt["mse"],
    "MAE": ckpt["mae"]
}
completed = 0

for p, h, d, name in grid_configs:
    path = grid_checkpoint_path(p, h, d, name)

    if path.exists():
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        grid_records[(p, h, d, name)] = {
            "Model": name,
            "PatchLen": p,
            "Horizon": h,
            "d_model": d,
            "MSE": ckpt["mse"],
            "MAE": ckpt["mae"]
        }
        completed += 1

save_grid_metrics(grid_records)

print("=== SURFACE PRESSURE 5-CHANNEL HEATMAP GRID SEARCH ===")
print("Device:", device)
print("Epochs per model:", epochs)
print("Total configurations:", len(grid_configs))
print("Completed:", completed)
print("Remaining:", len(grid_configs) - completed)
print("Results:", grid_results_dir)

# 6. Train missing configurations
for idx, (p, h, d, name) in enumerate(grid_configs, start=1):
    path = grid_checkpoint_path(p, h, d, name)

    if path.exists():
        print(
            f"[{idx}/{len(grid_configs)}] "
            f"SKIP {name} | P={p}, H={h}, d_model={d}"
        )
        continue

    print(
        f"\n[{idx}/{len(grid_configs)}] "
        f"{name} | P={p}, H={h}, d_model={d}"
    )

    # Build data for this horizon
    x_train, y_train, _ = build_pressure_dataset(
        raw, patch_len=p, horizon=h
    )

    # Create model
    model = create_model(
        name,
        patch_len=p,
        horizon=h,
        d_model=d
    )

    is_lstm = (name == "LSTM")

    # Train
    pred, loss, eig = train_pressure_model(
        model,
        x_train,
        y_train,
        epochs=epochs,
        lr=learning_rate,
        koop_attr=None if is_lstm else "koop",
        lyap_weight=0.0 if is_lstm else lyap_weight
    )

    # In-sample metrics
    truth = y_train.numpy()
    error = pred - truth

    mse_value = float(np.mean(error ** 2))
    mae_value = float(np.mean(np.abs(error)))

    # Save final checkpoint
    ckpt = {
        "model": name,
        "model_state_dict": model.cpu().state_dict(),
        "pred": torch.from_numpy(pred.copy()),
        "truth": torch.from_numpy(truth.copy()),
        "loss": torch.from_numpy(loss.copy()),
        "eig": torch.from_numpy(eig.copy()),
        "mse": mse_value,
        "mae": mae_value,
        "patch_len": p,
        "horizon": h,
        "d_model": d,
        "input_dim": input_dim,
        "epochs": epochs,
        "rng_torch": torch.get_rng_state(),
        "rng_cuda": torch.cuda.get_rng_state_all()
    }

    temp_path = path.with_suffix(".pt.tmp")
    torch.save(ckpt, temp_path)
    temp_path.replace(path)

    # Update metrics table
    grid_records[(p, h, d, name)] = {
        "Model": name,
        "PatchLen": p,
        "Horizon": h,
        "d_model": d,
        "MSE": mse_value,
        "MAE": mae_value
    }
    save_grid_metrics(grid_records)

    print(
        f"Saved {name}: "
        f"MSE={mse_value:.6f}, "
        f"MAE={mae_value:.6f}"
    )
    print(f"Checkpoint saved to: {path.resolve()}")

    del model, x_train, y_train, pred, loss, eig, ckpt
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== HEATMAP GRID SEARCH COMPLETE ===")
print("Completed runs:", len(grid_records))
print("Metrics saved:", grid_metrics_path)
print("Results directory:", grid_results_dir)

=== SURFACE PRESSURE 5-CHANNEL HEATMAP GRID SEARCH ===
Device: cuda
Epochs per model: 4000
Total configurations: 120
Completed: 42
Remaining: 78
Results: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\results\pressure_5channels_grid_e4000
[1/120] SKIP LSTM | P=120, H=10, d_model=8
[2/120] SKIP PatchTST | P=120, H=10, d_model=8
[3/120] SKIP Autoformer | P=120, H=10, d_model=8
[4/120] SKIP Informer | P=120, H=10, d_model=8
[5/120] SKIP LSTM | P=120, H=10, d_model=16
[6/120] SKIP PatchTST | P=120, H=10, d_model=16
[7/120] SKIP Autoformer | P=120, H=10, d_model=16
[8/120] SKIP Informer | P=120, H=10, d_model=16
[9/120] SKIP LSTM | P=120, H=10, d_model=24
[10/120] SKIP PatchTST | P=120, H=10, d_model=24
[11/120] SKIP Autoformer | P=120, H=10, d_model=24
[12/120] SKIP Informer | P=120, H=10, d_model=24
[13/120] SKIP LSTM | P=120, H=10, d_model=32
[14/120] SKIP PatchTST | P=120, H=10, d_model=32
[15/120] SKIP Autoformer | P=120, H=10, d_model=32
[16/120] SKIP Informer | P=120, H=10,